# Synthetic Data Generation

Goal: generate enough labeled stone and normal windows that model evaluation actually means something.

**Approach** — bootstrap from real data rather than building a from-scratch generative model:

1. **Harvesting library**: 600ms segments extracted from confirmed header-On periods, far from any stone event
2. **Stone impact library**: short audio snippets around each of the 10 confirmed metallic stone events — the actual transient sound of stone hitting blades
3. **Synthesis**:
   - Synthetic *normal* = harvesting segment + augmentation (amplitude scale, time shift, pitch shift, additive noise)
   - Synthetic *stone* = harvesting segment with a stone impact transient mixed in at a random pre-spike position + augmentation
4. Save everything to `data_synthetic/` as a pickle file for the next notebook to consume.

Why bootstrap rather than synthesise from scratch: with only 10 real stones, fitting a generative model on them would just memorise. Bootstrap-and-augment keeps the synthetic data anchored to real acoustic behaviour.

In [ ]:
from asammdf import MDF
import numpy as np
import pickle
import matplotlib.pyplot as plt
import librosa
from pathlib import Path
from scipy.stats import kurtosis

DATA_DIR     = Path("data")
MF4_FILES    = sorted(DATA_DIR.glob("*.mf4"))
OUT_DIR      = Path("data_synthetic")
OUT_DIR.mkdir(exist_ok=True)

SR              = 44100
VOLT_THRESHOLD  = 2000
MIN_SUSTAIN     = 5
WINDOW_BEFORE   = 0.5
WINDOW_AFTER    = 0.1
WINDOW_LEN      = 0.6
WIN_SAMPLES     = int(WINDOW_LEN * SR)
IMPACT_PRE      = 0.05   # 50ms before peak
IMPACT_POST     = 0.15   # 150ms after peak
IMPACT_SAMPLES  = int((IMPACT_PRE + IMPACT_POST) * SR)

rng = np.random.default_rng(42)
print("Setup complete. Window size:", WIN_SAMPLES, "samples")

In [ ]:
def get_episodes(status_channel):
    s = status_channel.samples.astype(str)
    t = status_channel.timestamps
    changes = np.where(s[:-1] != s[1:])[0]
    events = [(t[0], s[0])]
    for i in changes:
        events.append((t[i+1], s[i+1]))
    episodes, ep_start = [], None
    for ev_t, ev_s in events:
        if ev_s == 'On' and ep_start is None:
            ep_start = ev_t
        elif ev_s == 'Off' and ep_start is not None:
            episodes.append((ep_start, ev_t))
            ep_start = None
    if ep_start is not None:
        episodes.append((ep_start, t[-1]))
    return episodes

def get_stone_spike_times(volt_channel, episodes, threshold=VOLT_THRESHOLD, min_sustain=MIN_SUSTAIN):
    v, t = volt_channel.samples, volt_channel.timestamps
    spike_times = []
    for ep_start, ep_end in episodes:
        mask = (t >= ep_start) & (t <= ep_end)
        v_ep, t_ep = v[mask], t[mask]
        if len(v_ep) < min_sustain:
            continue
        above = v_ep > threshold
        sustained = np.zeros_like(above)
        count = 0
        for k in range(len(above)):
            if above[k]:
                count += 1
                if count >= min_sustain:
                    sustained[k - min_sustain + 1:k + 1] = True
            else:
                count = 0
        edges = np.diff(sustained.astype(int))
        onsets = t_ep[np.where(edges == 1)[0] + 1]
        for st in onsets:
            if not spike_times or st - spike_times[-1] > 1.0:
                spike_times.append(float(st))
    return spike_times

def extract_audio(audio_channel, center, before, after):
    t, s = audio_channel.timestamps, audio_channel.samples
    mask = (t >= center - before) & (t <= center + after)
    return s[mask].astype(np.float32)

## 1. Build the harvesting library and stone impact library

In [ ]:
harvesting_lib = []   # list of (run_name, audio_array[26460])
impact_lib     = []   # list of (run_name, audio_array[~8820]) — the impact transient

for f in MF4_FILES:
    mf     = MDF(f)
    audio  = mf.get("Sensor1")
    volt   = mf.get("VoltageSignal")
    status = mf.get("Status")

    episodes    = get_episodes(status)
    spike_times = get_stone_spike_times(volt, episodes)

    # Stone impact extraction — small window centred on the spike
    for st in spike_times:
        impact = extract_audio(audio, st, IMPACT_PRE, IMPACT_POST)
        if len(impact) >= IMPACT_SAMPLES * 0.9:
            impact_lib.append((f.stem, impact[:IMPACT_SAMPLES]))

    # Harvesting library — segments inside On periods, away from stones
    for ep_start, ep_end in episodes:
        if ep_end - ep_start < WINDOW_LEN + 4:
            continue
        # Walk the episode in steps of WINDOW_LEN+0.5s
        t = ep_start + 1
        while t + WINDOW_LEN + 0.5 < ep_end:
            if not any(abs(t + WINDOW_BEFORE - st) < 2.0 for st in spike_times):
                seg = extract_audio(audio, t + WINDOW_BEFORE, WINDOW_BEFORE, WINDOW_AFTER)
                if len(seg) >= WIN_SAMPLES * 0.9:
                    harvesting_lib.append((f.stem, seg[:WIN_SAMPLES]))
            t += WINDOW_LEN + 0.3

print(f"Harvesting library: {len(harvesting_lib)} segments  ({len(harvesting_lib)*WINDOW_LEN:.0f}s total)")
print(f"Stone impact library: {len(impact_lib)} transients ({len(impact_lib)*(IMPACT_PRE+IMPACT_POST):.1f}s total)")
print()
print("Harvesting segments per run:")
from collections import Counter
for r, n in Counter(r for r, _ in harvesting_lib).items():
    print(f"  {r[-10:]}: {n}")
print("\nStone impacts per run:")
for r, n in Counter(r for r, _ in impact_lib).items():
    print(f"  {r[-10:]}: {n}")

## 2. Visualise an impact — what we're inserting

Plot a real stone impact transient to sanity-check the extraction window covers the actual peak.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 6))
fig.suptitle("Real stone impact transients (extracted ±50/150ms around spike)", fontsize=11)

for i, (run, impact) in enumerate(impact_lib[:6]):
    t_axis = np.linspace(-IMPACT_PRE*1000, IMPACT_POST*1000, len(impact))
    ax = axes[i // 3, i % 3]
    ax.plot(t_axis, impact, lw=0.4, color='tomato')
    ax.axvline(0, color='red', lw=1, linestyle='--', alpha=0.5)
    ax.set_title(f"{run[-10:]}  (peak {np.abs(impact).max():.2f})", fontsize=9)
    ax.set_xlabel("Time relative to spike (ms)")
    ax.set_ylabel("Amplitude")

plt.tight_layout()
plt.show()

## 3. Augmentation functions

In [ ]:
def augment_audio(audio, rng_local):
    """Apply mild augmentations: amplitude, pitch (resample), additive noise."""
    a = audio.copy()

    # Amplitude scale ±20%
    a = a * rng_local.uniform(0.8, 1.2)

    # Pitch / speed shift via resampling (±5%)
    rate = rng_local.uniform(0.95, 1.05)
    if abs(rate - 1.0) > 0.01:
        new_len = int(len(a) / rate)
        a = np.interp(
            np.linspace(0, len(a) - 1, new_len),
            np.arange(len(a)), a
        ).astype(np.float32)
        # Pad or trim back to original length
        if len(a) >= len(audio):
            a = a[:len(audio)]
        else:
            a = np.pad(a, (0, len(audio) - len(a)))

    # Additive Gaussian noise (small)
    a = a + rng_local.normal(0, 0.005 * (np.std(a) + 1e-8),
                              size=len(a)).astype(np.float32)

    return a.astype(np.float32)


def synthesise_normal(rng_local):
    """Pick a random harvesting segment and augment it."""
    run, seg = harvesting_lib[rng_local.integers(0, len(harvesting_lib))]
    return augment_audio(seg, rng_local), run


def synthesise_stone(rng_local):
    """
    Build a synthetic stone window:
      1. Pick a random harvesting segment as base
      2. Pick a random stone impact transient
      3. Scale the impact (preserve impulsiveness, randomise amplitude)
      4. Mix it in at a random position inside the pre-spike zone
      5. Augment the whole window
    Returns: (audio, base_run, impact_run, spike_position_samples)
    """
    base_run, base = harvesting_lib[rng_local.integers(0, len(harvesting_lib))]
    impact_run, impact = impact_lib[rng_local.integers(0, len(impact_lib))]

    a = base.copy().astype(np.float32)
    imp = impact.copy().astype(np.float32)

    # Scale impact amplitude — preserve general magnitude but introduce variability
    imp *= rng_local.uniform(0.6, 1.3)

    # Slight stretch on the impact (resample, ±10%)
    rate = rng_local.uniform(0.9, 1.1)
    new_len = int(len(imp) / rate)
    imp = np.interp(
        np.linspace(0, len(imp) - 1, new_len),
        np.arange(len(imp)), imp
    ).astype(np.float32)

    # Decide where in the pre-spike zone to place the impact peak
    # — anywhere from 50ms before the window-end-of-pre-spike to 400ms before
    pre_samples = int(WINDOW_BEFORE * SR)
    impact_peak_offset = rng_local.integers(
        int(0.05 * SR),  # at least 50ms into the window
        pre_samples - int(IMPACT_POST * SR),  # leave room for tail
    )
    # The impact has its peak at IMPACT_PRE samples from its own start
    impact_start_in_window = impact_peak_offset - int(IMPACT_PRE * SR)
    impact_end_in_window   = impact_start_in_window + len(imp)

    # Clip to window bounds
    s_idx = max(0, impact_start_in_window)
    e_idx = min(len(a), impact_end_in_window)
    imp_s = max(0, -impact_start_in_window)
    imp_e = imp_s + (e_idx - s_idx)

    a[s_idx:e_idx] += imp[imp_s:imp_e]

    a = augment_audio(a, rng_local)
    return a, base_run, impact_run, impact_peak_offset

## 4. Generate the synthetic dataset

In [ ]:
N_STONE_SYNTH  = 200    # 20x more stones than real
N_NORMAL_SYNTH = 500    # ~8x more normal

synth_stones  = []   # (audio, base_source_run, impact_source_run, peak_offset)
synth_normals = []   # (audio, source_run)

rng_synth = np.random.default_rng(123)
for _ in range(N_STONE_SYNTH):
    audio, base_run, impact_run, peak = synthesise_stone(rng_synth)
    synth_stones.append((audio, base_run, impact_run, peak))

for _ in range(N_NORMAL_SYNTH):
    audio, source_run = synthesise_normal(rng_synth)
    synth_normals.append((audio, source_run))

print(f"Generated {len(synth_stones)} synthetic stone windows")
print(f"Generated {len(synth_normals)} synthetic normal windows")
print(f"Total dataset size now: {len(synth_stones) + len(synth_normals) + 10 + 63} windows")

## 5. Validate — do synthetic distributions match real?

If the synthetic data has obviously different statistics than the real data, the model will overfit to synthetic patterns. We need real and synthetic to look similar across the key features.

In [ ]:
def scalar_features(audio):
    n_pre = int(WINDOW_BEFORE * SR)
    pre = audio[:n_pre]
    w10 = int(SR * 0.01)
    rms_vals = [np.sqrt(np.mean(pre[i*w10:(i+1)*w10]**2)) for i in range(len(pre)//w10)]
    w20 = int(SR * 0.02)
    kurt_vals = [kurtosis(pre[i*w20:(i+1)*w20]) for i in range(len(pre)//w20)]
    return {
        'peak_rms':     max(rms_vals) if rms_vals else 0.0,
        'max_kurtosis': max(kurt_vals) if kurt_vals else 0.0,
    }

# Real stone/normal features
real_stone_feats  = [scalar_features(s) for _, s in [(r, s) for r, s in [(t[0], t[2]) for t in [(r2, st2, ex) for r2, st2, ex in [(rr, ss, extract_audio(MDF(DATA_DIR/(rr+'.mf4')).get('Sensor1'), ss, WINDOW_BEFORE, WINDOW_AFTER)) for rr, ss in [(r, st) for r in [f.stem for f in MF4_FILES] for st in get_stone_spike_times(MDF(DATA_DIR/(r+'.mf4')).get('VoltageSignal'), get_episodes(MDF(DATA_DIR/(r+'.mf4')).get('Status')))]] if len(ex) >= WIN_SAMPLES * 0.9]]]]

# (Above is dense — simpler version below)
real_stone_feats = []
real_normal_feats = []
rng_check = np.random.default_rng(42)

for f in MF4_FILES:
    mf = MDF(f)
    audio  = mf.get("Sensor1")
    volt   = mf.get("VoltageSignal")
    status = mf.get("Status")
    episodes = get_episodes(status)
    spikes   = get_stone_spike_times(volt, episodes)
    for st in spikes:
        w = extract_audio(audio, st, WINDOW_BEFORE, WINDOW_AFTER)
        if len(w) >= WIN_SAMPLES * 0.9:
            real_stone_feats.append(scalar_features(w[:WIN_SAMPLES]))
    for ep_start, ep_end in episodes:
        if ep_end - ep_start < WINDOW_LEN + 4:
            continue
        cands = rng_check.uniform(ep_start + 1, ep_end - WINDOW_LEN - 1, size=15)
        count = 0
        for ct in cands:
            if any(abs(ct - st) < 2.0 for st in spikes):
                continue
            w = extract_audio(audio, ct + WINDOW_BEFORE, WINDOW_BEFORE, WINDOW_AFTER)
            if len(w) >= WIN_SAMPLES * 0.9:
                real_normal_feats.append(scalar_features(w[:WIN_SAMPLES]))
                count += 1
            if count >= 3:
                break

synth_stone_feats  = [scalar_features(a) for a, _, _, _ in synth_stones]
synth_normal_feats = [scalar_features(a) for a, _ in synth_normals]

print(f"Real stones:   {len(real_stone_feats)}")
print(f"Real normals:  {len(real_normal_feats)}")
print(f"Synth stones:  {len(synth_stone_feats)}")
print(f"Synth normals: {len(synth_normal_feats)}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Real vs synthetic feature distributions", fontsize=11)

for ax, feat, title in zip(axes, ['peak_rms', 'max_kurtosis'],
                            ['Peak RMS', 'Max Kurtosis']):
    rs = [f[feat] for f in real_stone_feats]
    rn = [f[feat] for f in real_normal_feats]
    ss = [f[feat] for f in synth_stone_feats]
    sn = [f[feat] for f in synth_normal_feats]

    all_vals = np.concatenate([rs, rn, ss, sn])
    bins = np.linspace(np.percentile(all_vals, 1), np.percentile(all_vals, 99), 30)

    ax.hist(rn, bins=bins, alpha=0.45, color='steelblue', label=f'Real normal (n={len(rn)})', density=True)
    ax.hist(rs, bins=bins, alpha=0.7,  color='red',       label=f'Real stone  (n={len(rs)})', density=True)
    ax.hist(sn, bins=bins, alpha=0.3,  color='lightblue', label=f'Synth normal (n={len(sn)})', density=True,
             histtype='step', linewidth=1.5)
    ax.hist(ss, bins=bins, alpha=0.6,  color='darkorange', label=f'Synth stone (n={len(ss)})', density=True,
             histtype='step', linewidth=2)
    ax.set_title(title)
    ax.set_xlabel(feat)
    ax.set_ylabel("Density")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print("\nMedian comparison (real vs synthetic):")
for feat in ['peak_rms', 'max_kurtosis']:
    rs = np.median([f[feat] for f in real_stone_feats])
    ss = np.median([f[feat] for f in synth_stone_feats])
    rn = np.median([f[feat] for f in real_normal_feats])
    sn = np.median([f[feat] for f in synth_normal_feats])
    print(f"  {feat:13s}  stone:  real={rs:.4f}  synth={ss:.4f}  (ratio {ss/rs:.2f})")
    print(f"  {feat:13s}  normal: real={rn:.4f}  synth={sn:.4f}  (ratio {sn/rn:.2f})")

## 6. Save to disk

In [ ]:
out = {
    'synth_stones':  [(a.astype(np.float32), br, ir, p) for a, br, ir, p in synth_stones],
    'synth_normals': [(a.astype(np.float32), r) for a, r in synth_normals],
    'metadata': {
        'sr': SR,
        'window_samples': WIN_SAMPLES,
        'window_before_s': WINDOW_BEFORE,
        'window_after_s': WINDOW_AFTER,
        'n_stones': len(synth_stones),
        'n_normals': len(synth_normals),
        'real_runs_used': [f.stem for f in MF4_FILES],
    }
}

out_path = OUT_DIR / "synthetic_windows.pkl"
with open(out_path, 'wb') as fh:
    pickle.dump(out, fh, protocol=pickle.HIGHEST_PROTOCOL)

import os
print(f"Saved {out_path}")
print(f"Size: {os.path.getsize(out_path)/1024/1024:.1f} MB")

## Summary

Generated 200 synthetic stone windows and 500 synthetic normal windows by bootstrapping from real harvesting audio and real stone impact transients.

The next notebook (`model_training_synthetic.ipynb`) uses real data as held-out test and trains on real+synthetic to see whether the larger dataset actually moves the model metrics.